# 03 - Model Training, Benchmarking & Threshold Calibration
### Fraud Risk Intelligence & Detection Platform
This notebook covers:
1. Training a defensible baseline: Balanced L2-regularized Logistic Regression.
2. Training the primary supervised classifier: Gradient Boosted Trees (XGBoost) with `scale_pos_weight` and early stopping.
3. Training the unsupervised anomaly detector: Isolation Forest.
4. Precision-Recall AUC vs ROC-AUC evaluation under extreme imbalance (0.17%).
5. Operational threshold calibration targeting top-tier investigator capacity.
6. SHAP feature attribution (global importance and local risk explanations).


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.utils import load_artifact
from src.models import BaselineModel, XGBoostFraudModel, AnomalyDetector
from src.explainability import ModelExplainer
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score, confusion_matrix, classification_report
print("Imports loaded successfully.")


## 1. Load Preprocessed Feature Matrices & Labels


In [ ]:
train_df = pd.read_parquet("data/processed/train.parquet")
val_df = pd.read_parquet("data/processed/val.parquet")
test_df = pd.read_parquet("data/processed/test.parquet")

pipeline = load_artifact("artifacts/models/feature_pipeline.joblib")

X_train = pipeline.transform(train_df)
y_train = train_df["Class"].values.astype(int)

X_val = pipeline.transform(val_df)
y_val = val_df["Class"].values.astype(int)

X_test = pipeline.transform(test_df)
y_test = test_df["Class"].values.astype(int)

print(f"X_train: {X_train.shape} | Frauds: {y_train.sum():,}")
print(f"X_val:   {X_val.shape}   | Frauds: {y_val.sum():,}")
print(f"X_test:  {X_test.shape}  | Frauds: {y_test.sum():,}")


## 2. Load and Evaluate Baseline vs XGBoost


In [ ]:
baseline_model = load_artifact("artifacts/models/baseline_model.joblib")
xgb_model = load_artifact("artifacts/models/xgboost_model.joblib")

base_test_probs = baseline_model.predict_proba(pipeline.scaler.transform(X_test))
xgb_test_probs = xgb_model.predict_proba(X_test)

base_pr_auc = average_precision_score(y_test, base_test_probs)
xgb_pr_auc = average_precision_score(y_test, xgb_test_probs)
base_roc_auc = roc_auc_score(y_test, base_test_probs)
xgb_roc_auc = roc_auc_score(y_test, xgb_test_probs)

print("=" * 55)
print(f"Model Comparison on Test Set ({len(test_df):,} transactions):")
print("=" * 55)
print(f"Baseline Logistic Regression -> PR-AUC: {base_pr_auc:.4f} | ROC-AUC: {base_roc_auc:.4f}")
print(f"Primary XGBoost Classifier   -> PR-AUC: {xgb_pr_auc:.4f} | ROC-AUC: {xgb_roc_auc:.4f}")
print(f"Relative Improvement in PR-AUC: +{((xgb_pr_auc - base_pr_auc) / base_pr_auc):.2%}")


## 3. Threshold Calibration & Investigation Capacity
A default threshold of 0.50 is suboptimal for imbalanced fraud detection. We calibrate the threshold strictly on validation data to maximize F1 or optimize recall at an acceptable alert volume.


In [ ]:
calibrated_thresh = xgb_model.best_threshold
print(f"Calibrated Threshold: {calibrated_thresh:.4f}")

test_preds = (xgb_test_probs >= calibrated_thresh).astype(int)
print("\nClassification Report at Calibrated Threshold:")
print(classification_report(y_test, test_preds, digits=4))

cm = confusion_matrix(y_test, test_preds)
print(f"Confusion Matrix:\nTN: {cm[0,0]:,} | FP: {cm[0,1]:,}\nFN: {cm[1,0]:,}   | TP: {cm[1,1]:,}")


## 4. Global SHAP Feature Importances
TreeExplainer computes exact Shapley values to identify primary risk drivers.


In [ ]:
shap_csv = Path("artifacts/reports/shap_global_importance.csv")
if shap_csv.exists():
    shap_df = pd.read_csv(shap_csv)
    print("Top 10 Global Risk Drivers by Mean |SHAP| Value:")
    print(shap_df.head(10).to_string(index=False))


## 5. Unsupervised Anomaly Scoring (Isolation Forest)


In [ ]:
anomaly_model = load_artifact("artifacts/models/anomaly_detector.joblib")
anomaly_scores = anomaly_model.score_anomaly(pipeline.scaler.transform(X_test))

print("Isolation Forest Normalized Anomaly Scores [0.0 - 1.0]:")
print(pd.Series(anomaly_scores).describe().round(4))
print(f"\nMean anomaly score for Legitimate txs: {anomaly_scores[y_test == 0].mean():.4f}")
print(f"Mean anomaly score for Fraudulent txs: {anomaly_scores[y_test == 1].mean():.4f}")
